# Cribl API (%%cribl_api)

Use the **%%cribl_api** cell magic to call the [Cribl REST API](https://docs.cribl.io/cribl-as-code/api/) from a code cell. The first line is the **HTTP method**, the **path** (under your deployment's `CRIBL_API_URL`), and options such as **`var=`** and **`response=`**. The rest of the cell is **YAML** describing `headers` and, for writes, a **`json`** object for the request body.

In the editor, **Tab** after the method opens path suggestions; **hover** the method and path on the first line for a short description when the path is in the app’s catalog. Accepting a **POST** with an empty YAML block can insert a sample `json` body to edit.

This only works when the app runs **inside the Cribl platform** (a real `CRIBL_API_URL`). Local `npm run dev` has no API base, so these cells will error until you use a hosted app.


## First line (after `%%cribl_api`)

- **`GET|POST|PUT|PATCH|DELETE`** then **`/path...`** — path must start with `/` (e.g. Search context: `/m/default_search/...`).
- **`var=name`** — Python variable for the HTTP response (default `cribl_api_result`).
- **`preview=true|false`** — print a short preview in the output (default `true`).
- **`response=json|raw|text`** — `json` parses the body into a dict/list; `raw` and `text` keep the body as a string.
- **`template=auto|on|off`** — same as `%%cribl_search`: Jinja2 can expand `{{ name }}` in the YAML from notebook variables before the request is sent.


## YAML body (lines 2+)

Use a small mapping with only these top-level keys:

- **`headers:`** — optional; string values for request headers.
- **`json:`** — optional; object serialized to the JSON request body; `Content-Type: application/json` is set if you omit it.
- **`body:`** — optional raw string body (only when you are not using `json:`).

If both `json` and `body` are present, **`json` wins**.


## List search jobs (GET)

Search API routes use the `/m/default_search` group (see the Cribl API docs).


In [ ]:
%%cribl_api GET /m/default_search/search/jobs var=search_jobs preview=true response=json
headers:
  Accept: application/json


## Global settings (GET)

Some endpoints are **global** (not under a worker group or `default_search`). Example: git settings.


In [ ]:
%%cribl_api GET /system/settings/git-settings var=git_settings preview=true response=json


## Create a search job (POST)

Submits a job with the same JSON fields the API expects (query, time range, etc.). This **runs a real job** in your environment—use a narrow time window in shared systems.


In [ ]:
%%cribl_api POST /m/default_search/search/jobs var=job_post_result preview=true response=json
json:
  query: |
    cribl dataset="cribl_search_sample" | limit 100
  earliest: "-1h"
  latest: "now"
  sampleRate: 1


## Jinja in the YAML block

Set Python values, then reference them in the YAML with `{{ var }}` (with `template=auto` or `on`). The block is rendered in the **kernel** before the YAML is parsed.


In [ ]:
# Used by the next cell
api_dataset = "cribl_search_sample"
api_query_limit = 50


In [ ]:
%%cribl_api POST /m/default_search/search/jobs var=jinja_job response=json template=on
json:
  query: |
    cribl dataset="{{ api_dataset }}" | limit {{ api_query_limit }}
  earliest: "-1h"
  latest: "now"
  sampleRate: 1


## Inspecting the raw body

Use **`response=raw`** (or `text`) when you want the HTTP body as a **string** in Python, for example when debugging the exact bytes returned by the server.


In [ ]:
%%cribl_api GET /m/default_search/search/jobs var=jobs_raw preview=false response=raw
headers:
  Accept: application/json


For **Cribl Search queries** in KQL (without hand-crafting REST calls), see **`Cribl_Search_Examples.ipynb`** and the **%%cribl_search** magic.
